In [26]:
import pandas as pd
import numpy as np

FILES = {
    "strict_mbu":             "data/raw/claims_strict_mbu.csv",
    "regular_sbu":             "data/raw/claims_regular_sbu.csv",
    "lenient_lbu":             "data/raw/claims_lenient_lbu.csv",
    "strict_lbu":              "data/raw/claims_strict_lbu.csv",
    "regular_mbu":             "data/raw/claims_regular_mbu.csv",
    "lenient_sbu":             "data/raw/claims_lenient_sbu.csv",
    "regular_mbu_highvolume":  "data/raw/claims_regular_mbu_highvolume.csv",
}

RAW = {}
for name, path in FILES.items():
    RAW[name] = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=["", "NULL", "null"])
for name, df in RAW.items():
    print(f"{name:26} rows={len(df):6}  cols={df.shape[1]}")

strict_mbu                 rows=  1159  cols=45
regular_sbu                rows=   520  cols=45
lenient_lbu                rows=  2391  cols=45
strict_lbu                 rows=  2277  cols=45
regular_mbu                rows=  1075  cols=45
lenient_sbu                rows=   465  cols=45
regular_mbu_highvolume     rows=  5130  cols=45


In [27]:
cols = RAW["regular_mbu"].columns.tolist()
assert all(list(df.columns) == cols for df in RAW.values()), \
    "column mismatch between datasets"

print(f"{len(cols)} columns, identical across all seven:\n")
for i, c in enumerate(cols):
    print(f"{i:2}  {c}")

45 columns, identical across all seven:

 0  item_pk
 1  item_id
 2  report_pk
 3  employee_id
 4  report_id
 5  expense_type_id
 6  expense_category_code
 7  bill_number
 8  bill_date
 9  bill_amount
10  bill_currency_id
11  overridden_amount
12  mileage_unit
13  mileage_quantity
14  mileage_rate_per_unit
15  mileage_original_rate_per_unit
16  mileage_overridden_quantity
17  item_status
18  report_status
19  auto_approved
20  approval_date
21  rejection_date
22  policy_id
23  rule_id
24  limit_per_claim
25  limit_per_day
26  limit_per_month
27  limit_per_quarter
28  limit_per_year
29  allow_beyond_limit
30  frequency_per_day
31  frequency_per_month
32  frequency_per_quarter
33  frequency_per_year
34  allow_beyond_frequency
35  configured_reviewer_chain
36  configured_reviewer_count
37  remark_count
38  last_workflow_action
39  is_duplicate_flagged
40  duplicate_match_count
41  override_count
42  item_created_on
43  item_updated_on
44  submission_date


In [28]:
pd.set_option("display.max_rows", 100)
pd.DataFrame({
    "nulls":    df.isna().sum(),
    "null_%":   (df.isna().mean() * 100).round(1),
    "distinct": df.nunique(),
})

,nulls,null_%,distinct
item_pk,0,0.0,5130
item_id,0,0.0,5130
report_pk,0,0.0,3362
employee_id,0,0.0,25
report_id,0,0.0,3362
expense_type_id,0,0.0,15
expense_category_code,0,0.0,6
bill_number,1552,30.3,3565
bill_date,0,0.0,557
bill_amount,0,0.0,4969


In [29]:
DATE_COLS = [
    "bill_date", "submission_date", "approval_date", "rejection_date",
    "item_created_on", "item_updated_on",
]

NUM_COLS = [
    "bill_amount", "overridden_amount",
    "mileage_quantity", "mileage_rate_per_unit",
    "mileage_original_rate_per_unit", "mileage_overridden_quantity",
    "limit_per_claim", "limit_per_day", "limit_per_month",
    "limit_per_quarter", "limit_per_year",
    "frequency_per_day", "frequency_per_month",
    "frequency_per_quarter", "frequency_per_year",
    "duplicate_match_count", "override_count", "remark_count",
    "configured_reviewer_count",
]

BOOL_COLS = ["auto_approved", "is_duplicate_flagged",
             "allow_beyond_limit", "allow_beyond_frequency"]

ID_COLS = ["item_pk", "report_pk", "employee_id", "bill_currency_id",
           "expense_type_id"]


def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in DATE_COLS:
        df[c] = pd.to_datetime(df[c], errors="coerce")

    for c in NUM_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    for c in BOOL_COLS:
        df[c] = df[c].map({"TRUE": True, "True": True, "true": True,
                           "FALSE": False, "False": False, "false": False})

    for c in ID_COLS:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    df["bill_number"] = df["bill_number"].str.strip()
    df["expense_category_code"] = df["expense_category_code"].astype("category")

    return df


TYPED = {name: cast_types(df) for name, df in RAW.items()}

print(TYPED["regular_mbu"].dtypes.value_counts())
TYPED["regular_mbu"][["item_id", "employee_id", "expense_category_code",
                       "expense_type_id", "bill_amount", "item_status",
                       "bill_date", "submission_date"]].head()


float64           12
object            10
int64              7
datetime64[ns]     6
Int64              5
bool               4
category           1
Name: count, dtype: int64


,item_id,employee_id,expense_category_code,expense_type_id,bill_amount,item_status,bill_date,submission_date
0,TRVL-501,1000,TRVL,7,6136.25,Paid,2025-07-15,2025-07-29 00:00:00.000000000
1,MEAL-502,1000,MEAL,4,2954.83,Accepted,2025-01-30,2025-02-01 00:00:00.000000000
2,MEAL-503,1000,MEAL,4,3211.11,Accepted,2025-03-17,2025-03-24 23:45:02.071124091
3,MEAL-504,1000,MEAL,4,3369.99,Accepted,2025-04-29,2025-05-01 02:46:20.405393764
4,MEAL-505,1000,MEAL,4,3105.88,Paid,2025-06-23,2025-06-25 11:03:04.776940889


In [30]:
def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # label--> approval
    df["is_approved"] = df["item_status"].isin(["Accepted", "Paid"])
    # label--> rejected
    df["is_rejected"] = df["item_status"] == "Rejected"

    # a CLEAN approval
    df["is_clean_approval"] = (
        df["is_approved"]
        & (df["auto_approved"] != True)
        & (df["override_count"].fillna(0) == 0)
    )

    # timing
    df["submit_lag_days"] = (df["submission_date"] - df["bill_date"]).dt.days
    df["bill_period"] = df["bill_date"].dt.to_period("M")

    # mileage claims
    df["is_mileage"] = df["mileage_quantity"].notna()

    return df


FINAL = {name: add_derived(df) for name, df in TYPED.items()}

for name, df in FINAL.items():
    print(f"{name:26} approved={df.is_approved.sum():5}  "
          f"rejected={df.is_rejected.sum():5}  "
          f"clean={df.is_clean_approval.sum():5}  "
          f"mileage={df.is_mileage.sum():4}")

strict_mbu                 approved=  877  rejected=  282  clean=  827  mileage= 138
regular_sbu                approved=  468  rejected=   52  clean=  450  mileage=  61
lenient_lbu                approved= 2359  rejected=   32  clean= 2306  mileage= 357
strict_lbu                 approved= 1715  rejected=  562  clean= 1634  mileage= 329
regular_mbu                approved=  966  rejected=  109  clean=  930  mileage= 215
lenient_sbu                approved=  460  rejected=    5  clean=  429  mileage=  53
regular_mbu_highvolume     approved= 4618  rejected=  512  clean= 4296  mileage= 891


In [32]:
def structural_checks(name, df):
    clean = df[df.is_clean_approval]

    mixed_reports = df.groupby("report_pk")["item_status"].nunique()

    pairs_type = clean.groupby(
        ["employee_id", "expense_category_code", "expense_type_id"],
        observed=True
    ).size()

    pairs_cat = clean.groupby(
        ["employee_id", "expense_category_code"],
        observed=True
    ).size()

    return dict(
        dataset=name,
        rows=len(df),
        employees=df.employee_id.nunique(),
        reject_rate=f"{(df.item_status == 'Rejected').mean():.1%}",
        auto_approved=int(df.auto_approved.sum()),
        clean_approvals=int(len(clean)),
        eligible_pairs_type=f"{(pairs_type >= 3).sum()}/{len(pairs_type)} "
                             f"({(pairs_type>=3).mean():.0%})",
        eligible_pairs_category=f"{(pairs_cat >= 3).sum()}/{len(pairs_cat)} "
                                 f"({(pairs_cat>=3).mean():.0%})",
        duplicate_flagged=int(df.is_duplicate_flagged.sum()),
        overridden=int((df.override_count > 0).sum()),
    )

summary = pd.DataFrame([structural_checks(name, df) for name, df in FINAL.items()])
summary

,dataset,rows,employees,reject_rate,auto_approved,clean_approvals,eligible_pairs_type,eligible_pairs_category,duplicate_flagged,overridden
0,strict_mbu,1159,25,24.3%,31,827,56/87 (64%),52/70 (74%),5,19
1,regular_sbu,520,10,10.0%,2,450,28/39 (72%),21/30 (70%),2,16
2,lenient_lbu,2391,50,1.3%,32,2306,124/192 (65%),113/160 (71%),17,21
3,strict_lbu,2277,50,24.7%,48,1634,104/162 (64%),98/134 (73%),18,33
4,regular_mbu,1075,25,10.1%,12,930,65/96 (68%),61/82 (74%),6,24
5,lenient_sbu,465,10,1.1%,28,429,23/38 (61%),21/30 (70%),1,3
6,regular_mbu_highvolume,5130,25,10.0%,179,4296,156/194 (80%),116/129 (90%),26,143


In [33]:
def eligibility_by_category(name, df):
    clean = df[df.is_clean_approval]
    pairs = clean.groupby(
        ["employee_id", "expense_category_code", "expense_type_id"],
        observed=True
    ).size().reset_index(name="n_claims")

    out = (pairs.groupby("expense_category_code", observed=True)
                .agg(pairs=("n_claims", "size"),
                     eligible=("n_claims", lambda s: (s >= 3).sum()))
                .reset_index())
    out["eligible_pct"] = (out["eligible"] / out["pairs"]).map("{:.0%}".format)
    out.insert(0, "dataset", name)
    return out[["dataset", "expense_category_code", "pairs", "eligible", "eligible_pct"]]

per_category = pd.concat(
    [eligibility_by_category(name, df) for name, df in FINAL.items()],
    ignore_index=True
)
per_category


,dataset,expense_category_code,pairs,eligible,eligible_pct
0,strict_mbu,FUEL,11,7,64%
1,strict_mbu,INT,8,6,75%
2,strict_mbu,LEARN,20,7,35%
3,strict_mbu,MEAL,17,12,71%
4,strict_mbu,MED,11,8,73%
5,strict_mbu,TRVL,20,16,80%
6,regular_sbu,FUEL,5,3,60%
7,regular_sbu,INT,5,4,80%
8,regular_sbu,LEARN,9,5,56%
9,regular_sbu,MEAL,13,12,92%


In [34]:
combined = pd.concat(
    [df.assign(dataset=name) for name, df in FINAL.items()],
    ignore_index=True
)

print("combined rows:", len(combined))
print(combined.groupby("dataset")["employee_id"].apply(lambda s: s.min()).to_dict())

combined rows: 13017
{'lenient_lbu': 1000, 'lenient_sbu': 1000, 'regular_mbu': 1000, 'regular_mbu_highvolume': 1000, 'regular_sbu': 1000, 'strict_lbu': 1000, 'strict_mbu': 1000}


In [35]:
from ydata_profiling import ProfileReport

profile = ProfileReport(
    combined, minimal=True,
    title="Synthetic Claims — All Seven Datasets"
)
profile.to_file("reports/synthetic_all_seven_profile.html")

Summarize dataset:   2%|  | 1/57 [00:00<00:20,  2.72it/s, Describe variable: bill_date]
Summarize dataset:  18%|▏| 10/57 [00:00<00:01, 26.81it/s, Describe variable: item_statu
Summarize dataset:  39%|▍| 22/57 [00:00<00:00, 44.86it/s, Describe variable: allow_beyoC:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  39%|▍| 22/57 [00:00<00:00, 44.86it/s, Describe variable: allow_beyoC:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseco

In [36]:
ProfileReport(FINAL["regular_mbu_highvolume"], minimal=True,
              title="regular_mbu_highvolume").to_file(
    "reports/regular_mbu_highvolume_profile.html"
)

Summarize dataset:   9%| | 5/56 [00:00<00:02, 17.06it/s, Describe variable: mileage_uni
Summarize dataset:  34%|▎| 19/56 [00:00<00:01, 33.63it/s, Describe variable: limit_per_
Summarize dataset:  36%|▎| 20/56 [00:00<00:01, 33.63it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
Summarize dataset:  36%|▎| 20/56 [00:00<00:01, 33.63it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  43%|▍| 24/56 [00:00<00:00, 33.63it/s, Describe variable: frequency_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_prof

In [37]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["regular_mbu"], minimal=True,
              title="regular_mbu").to_file(
    "reports/regular_mbu_profile.html"
)

Summarize dataset:  11%| | 6/56 [00:00<00:01, 40.05it/s, Describe variable: mileage_qua
Summarize dataset:  36%|▎| 20/56 [00:00<00:00, 85.18it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  38%|▍| 21/56 [00:00<00:00, 85.18it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\App

In [38]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["regular_sbu"], minimal=True,
              title="regular_sbu").to_file(
    "reports/regular_sbu_profile.html"
)

Summarize dataset:  14%|▏| 8/56 [00:00<00:02, 16.71it/s, Describe variable: mileage_ori
Summarize dataset:  38%|▍| 21/56 [00:00<00:00, 44.21it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\des

In [39]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["strict_mbu"], minimal=True,
              title="strict_mbu").to_file(
    "reports/strict_mbu_profile.html"
)

Summarize dataset:  11%| | 6/56 [00:00<00:04, 11.55it/s, Describe variable: mileage_qua
Summarize dataset:  34%|▎| 19/56 [00:00<00:00, 48.57it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  43%|▍| 24/56 [00:00<00:00, 45.89it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
Summarize dataset:

In [40]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["strict_lbu"], minimal=True,
              title="strict_lbu").to_file(
    "reports/strict_lbu_profile.html"
)

Summarize dataset:  11%| | 6/56 [00:00<00:04, 10.60it/s, Describe variable: mileage_qua
Summarize dataset:  36%|▎| 20/56 [00:00<00:00, 41.43it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  41%|▍| 23/56 [00:00<00:00, 41.43it/s, Describe variable: frequency_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\App

In [41]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["lenient_lbu"], minimal=True,
              title="lenient_lbu").to_file(
    "reports/lenient_lbu_profile.html"
)

Summarize dataset:  12%|▏| 7/56 [00:00<00:05,  9.58it/s, Describe variable: mileage_rat
Summarize dataset:  34%|▎| 19/56 [00:00<00:00, 39.58it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
Summarize dataset:  41%|▍| 23/56 [00:00<00:00, 36.40it/s, Describe variable: allow_beyoC:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\App

In [42]:
from ydata_profiling import ProfileReport
ProfileReport(FINAL["lenient_sbu"], minimal=True,
              title="lenient_sbu").to_file(
    "reports/lenient_sbu_profile.html"
)

Summarize dataset:  23%|▏| 13/56 [00:01<00:09,  4.62it/s, Describe variable: mileage_ov
Summarize dataset:  30%|▉  | 17/56 [00:01<00:07,  5.34it/s, Describe variable: rule_id]
Summarize dataset:  36%|▎| 20/56 [00:01<00:05,  6.14it/s, Describe variable: limit_per_C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:58: UserWarning: Discarding nonzero nanoseconds in conversion.
  "max": pd.Timestamp.to_pydatetime(series.max()),
C:\Users\sridh\AppData\Local\Programs\Python\Python310\lib\site-packages\ydata_profiling\model\pandas\describe_date_pandas.py:57: UserWarning: Discarding nonzero nanoseconds in conversion.
  "min": pd.Timestamp.to_pydatetime(series.min()),
C:\Users\sridh\Ap

In [43]:
import os
os.makedirs("data/interim", exist_ok=True)

for name, df in FINAL.items():
    path = f"data/interim/claims_{name}_typed.parquet"
    df.to_parquet(path, index=False)
    print(f"saved {path}  ({len(df)} rows)")

saved data/interim/claims_strict_mbu_typed.parquet  (1159 rows)
saved data/interim/claims_regular_sbu_typed.parquet  (520 rows)
saved data/interim/claims_lenient_lbu_typed.parquet  (2391 rows)
saved data/interim/claims_strict_lbu_typed.parquet  (2277 rows)
saved data/interim/claims_regular_mbu_typed.parquet  (1075 rows)
saved data/interim/claims_lenient_sbu_typed.parquet  (465 rows)
saved data/interim/claims_regular_mbu_highvolume_typed.parquet  (5130 rows)
